# 01 - Download the 10-K filings from SEC EDGAR

Steps: download the latest two 10-K filings of 19 companies from EDGAR (the SEC's free public archive), cut out
the three sections we use (Item 1 Business, Item 1A Risk Factors, Item 7 MD&A), check that every section was found,
and upload the result to a **private** Hugging Face dataset, so the next notebooks can start from it.

No GPU needed: use a normal (CPU) runtime. It takes a few minutes because EDGAR allows only a few requests per second.

**Before running:** add two Colab Secrets (key icon on the left, "Notebook access" on):
- `EDGAR_USER_AGENT`: your name and e-mail, for example `Jane Doe jane@example.com` (the SEC asks every client to
  say who it is; the value stays in your Secrets and is never written in the repository);
- `HF_TOKEN`: a Hugging Face token with write access.

In [ ]:
import os
import subprocess
import sys

REPO = "10K-agentic-rag"
BRANCH = "main"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://github.com/ilaria-cesa/{REPO}.git"], check=True
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements-base.txt

In [ ]:
from google.colab import userdata

os.environ["EDGAR_USER_AGENT"] = userdata.get("EDGAR_USER_AGENT")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
HF_REPO = "IlariaCesa/10k-agentic-rag-data"

Download and section extraction. The command is resumable: files already on disk are not downloaded again.
Companies are listed in `src/ingest/companies.py`.

In [ ]:
from src.ingest.run import main as download

download([])  # default: the 19 companies, latest 2 filings each, files in data/raw and data/parsed

Health check: one row per filing, with the number of characters found in each section.
Every section should have at least 2000 characters (`ok` = True), and every company should have 2 filings.

In [ ]:
from pathlib import Path

import pandas as pd

from src.ingest.summary import build_rows, health_report

rows = build_rows(Path("data/parsed"))
table = pd.DataFrame(rows)
print("\n".join(health_report(rows)))
assert len(rows) == 38 and table["ok"].all(), "something is missing: look at the table below"
table

Upload to the private dataset repo (raw HTML and extracted sections).

In [ ]:
from src.hub.dataset import upload_folder

for name in ["raw", "parsed"]:
    print(upload_folder(HF_REPO, Path("data") / name, name))

Next: `02a_chunk_and_embed_text.ipynb` starts from the `parsed/` folder of this dataset.